# DPO v2 of mini-GPT on Kaggle

Required notebook settings: accelerator **GPU T4**, **Internet on**, and the dataset attached. It must contain `dpo_prompts.jsonl`, `tokenizer.json` and `sft_best.pt` (the SFT checkpoint).

Compared with v1: preference pairs are filtered for naturalness, the loss adds a likelihood term on the preferred story, and the optimization is gentler (higher beta, lower learning rate, fewer steps) with periodic snapshots.

In [ ]:
import os

REPO_URL = "https://github.com/bamont/minigpt.git"
REPO_DIR = "/kaggle/working/minigpt"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

In [ ]:
DATA_DIR = "/kaggle/input/datasets/bbbaaammm/minigpt-tinystories/minigpt-tinystories"
PAIRS_DIR = "/kaggle/working/dpo_v2_data"
RUN_DIR = "/kaggle/working/runs/dpo_v2"

print(os.listdir(DATA_DIR))

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())
!nvidia-smi

In [ ]:
!python scripts/build_preference_pairs.py --prompts {DATA_DIR}/dpo_prompts.jsonl --checkpoint {DATA_DIR}/sft_best.pt --tokenizer {DATA_DIR}/tokenizer.json --output-dir {PAIRS_DIR} --samples-per-prompt 6 --batch-size 132

In [ ]:
!python -m minigpt.training.dpo --config configs/dpo_v2.yaml --resume --override data.train={PAIRS_DIR}/dpo_train.npz data.val={PAIRS_DIR}/dpo_val.npz data.tokenizer={DATA_DIR}/tokenizer.json init.checkpoint={DATA_DIR}/sft_best.pt train.out_dir={RUN_DIR}

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{RUN_DIR}/metrics.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]
train = [r for r in records if r["type"] == "train"]
evals = [r for r in records if r["type"] == "eval"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot([r["step"] for r in train], [r["loss"] for r in train], label="train")
axes[0].plot([r["step"] for r in evals], [r["val_loss"] for r in evals], label="validation")
axes[0].set_xlabel("step")
axes[0].set_ylabel("total loss")
axes[0].legend()
axes[1].plot([r["step"] for r in evals], [r["val_accuracy"] for r in evals])
axes[1].set_xlabel("step")
axes[1].set_ylabel("validation preference accuracy")
axes[2].plot([r["step"] for r in evals], [r["val_nll"] for r in evals])
axes[2].set_xlabel("step")
axes[2].set_ylabel("validation NLL of preferred stories")
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/curves.png", dpi=150)
plt.show()

print(sorted(f for f in os.listdir(RUN_DIR) if f.endswith(".pt")))